# Unknown-format CRF digitizer - code-generation induction

For every CRF PDF in the input managed folder - **any vendor format, no template,
no configuration** - a pure-Python pass clusters the pages by structural layout
(word-blind typography tokens, per-document chrome damping, similarity threshold
self-selected per document) and picks a handful of representative pages (~4-12);
the LLM (via LLM Mesh) writes a document-specific Python parser from those pages;
the parser runs in a sandboxed subprocess over the whole document; machine gates
plus an LLM page-grounded audit challenge the result; and the LLM revises its own
code in a bounded loop. Output per document: `(form_name, field_name, page)`
records. OIDs are resolved downstream by name mapping against the rule library.

**Requirements**
- Code env for this notebook (the sandbox subprocess uses the same interpreter):
  `PyMuPDF`, `pandas`; `rapidfuzz` optional (mapping cell only).
- ONE managed folder (`INPUT_FOLDER` = `OUTPUT_FOLDER` below) holding:
  the pipeline module `.py` files under `code/` (upload them from the repo's
  `experiments/recipe_prototype/dataiku/folder_code/`), and the input CRF PDFs
  anywhere else in the folder. All artifacts are written back to it.
- LLM Mesh: uses the project variable `default_llm_model` (Claude Sonnet 4.5)
  unless `LLM_ID` overrides it below.

**LLM budget - bounded, never endless.** Per document: at most `MAX_VERSIONS`
parser versions (1 generation call each, +1 audit call when gates pass, +1 audit
reprompt if a reply is malformed/partial) plus at most one coverage-confirmation
call. Defaults give a worst case of ~16 calls per document; the benchmark runs
took 7-10. The loop stops early on a clean audit (converged) or when a version
fails to improve on the previous one (diminishing returns; two consecutive
gate-failed versions keep revising until the cap); the best-scoring version -
never merely the last - is exported.

*This notebook is generated by `experiments/recipe_prototype/build_dataiku_notebook.py`.
The pipeline code is NOT embedded here: the bootstrap cell downloads it from the
managed folder's `code/` subpath. To change pipeline behavior, edit the repo
modules and re-upload them from `dataiku/folder_code/` - regenerate this notebook
only when cell logic changes.*


In [ ]:
# ------------------------------- CONFIG -------------------------------------
# ONE managed folder for everything - the same folder id the io-capture
# notebook (ecs_io_capture_384_201_00002) persists to. It holds:
#   code/<module>.py   the pipeline modules (upload from dataiku/folder_code/)
#   *.pdf              the input CRFs (root or any subpath)
#   <doc_key>/...      artifacts written back by this notebook
# Safe to share because the PDF fetch only picks *.pdf (code/ is skipped) and
# the artifact upload writes no PDFs.
INPUT_FOLDER = '3UkrB0N9'            # managed folder (name or id) with input CRF PDFs
OUTPUT_FOLDER = '3UkrB0N9'           # managed folder (name or id) for all artifacts
CODE_SUBPATH = 'code'                # subpath in that folder holding the module .py files
LLM_ID = None                        # None -> project variable 'default_llm_model'
COMPLETION_SETTINGS = {'temperature': 0.2, 'maxOutputTokens': 8000}  # best effort

MAX_VERSIONS = 5     # hard cap on parser versions per document
DOC_FILTER = ''      # substring filter on PDF names ('' = all)
MAX_DOCS = None      # int -> cap the number of documents (smoke runs)

UPLOAD_PAGE_PNGS = False   # representative-page PNGs are handy but heavy
RUN_OID_MAPPING = False    # name->OID funnel + LLM ranker (see mapping cell)
ECS_INDEX_DATASET = 'ecs_index_data'  # dataset with form_field_value / variable_name

WORK_DIR = None      # None -> ./crf_codegen_work under the kernel's cwd


In [ ]:
# Bootstrap: download the pipeline modules from the managed folder's code/
# subpath into a local work dir (no repo checkout on the DSS host). The work
# dir is plain local scratch - created here, disposable, re-runnable.
import os
import sys

import dataiku

WORK = os.path.abspath(WORK_DIR or 'crf_codegen_work')
MODULES_DIR = os.path.join(WORK, 'modules')
os.makedirs(MODULES_DIR, exist_ok=True)
# pipeline paths (input staging dir, output dir) derive from this env var;
# it must be set BEFORE the modules are imported
os.environ['ECS_BASE'] = WORK

EXPECTED_MODULES = ['common.py', 'generic_profile.py', 'replay.py', 'induction.py', 'codegen.py', 'sandbox_runner.py', 'stage0_cluster.py', 'run_cli_induction.py', 'oid_mapping.py']

_folder = dataiku.Folder(INPUT_FOLDER)
_prefix = '/' + CODE_SUBPATH.strip('/') + '/'
_found = {}
for _p in _folder.list_paths_in_partition():
    if ('/' + _p.lstrip('/')).startswith(_prefix) and _p.endswith('.py'):
        _found[_p.rsplit('/', 1)[-1]] = _p

_missing = sorted(set(EXPECTED_MODULES) - set(_found))
if _missing:
    raise RuntimeError(
        'pipeline modules missing from folder ' + str(INPUT_FOLDER) + ' under '
        + _prefix + ' : ' + ', '.join(_missing)
        + ' - upload them from the repo dir experiments/recipe_prototype/dataiku/folder_code/')

for _name in EXPECTED_MODULES:
    with _folder.get_download_stream(_found[_name]) as _s:
        _src = _s.read()
    with open(os.path.join(MODULES_DIR, _name), 'wb') as _f:
        _f.write(_src)
    print('module fetched:', _name, '(' + str(len(_src)) + ' bytes)')


In [ ]:
# Import the pipeline from the materialized modules. Pop-and-import (never
# importlib.reload): reload would re-execute a module from wherever it was FIRST
# imported, so a foreign 'codegen'/'common' already living in the kernel would
# silently shadow the bundle. Re-run safe top to bottom.
_PIPELINE_MODULES = ('common', 'generic_profile', 'replay', 'induction',
                     'codegen', 'stage0_cluster', 'run_cli_induction')
if MODULES_DIR not in sys.path:
    sys.path.insert(0, MODULES_DIR)
for _m in _PIPELINE_MODULES:
    sys.modules.pop(_m, None)

import common
import codegen
import stage0_cluster
import run_cli_induction as rci

for _m in _PIPELINE_MODULES:
    _f = os.path.abspath(getattr(sys.modules[_m], '__file__', '') or '')
    assert _f.startswith(os.path.abspath(MODULES_DIR) + os.sep),         _m + ' imported from ' + _f + ' instead of the bundle - check sys.path'

from common import CRF_DIR, OUT_DIR, doc_key, list_root_pdfs

os.makedirs(CRF_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
print('work dir:', WORK)


In [ ]:
# Stage the input PDFs from the managed folder into the local work dir
# (PyMuPDF needs real files; managed folders may be S3-backed). PDFS - the list
# bound HERE - is what every later cell iterates: a warm work dir from an
# earlier session never adds documents the current DOC_FILTER excludes.
import re
import shutil

import dataiku

folder_in = dataiku.Folder(INPUT_FOLDER)
paths = sorted(p for p in folder_in.list_paths_in_partition() if p.lower().endswith('.pdf'))
if DOC_FILTER:
    paths = [p for p in paths if DOC_FILTER.lower() in p.lower()]
if MAX_DOCS:
    paths = paths[:MAX_DOCS]

flat = {}
for p in paths:
    local_name = re.sub(r'[\\/]+', '_', p.strip('/'))
    if local_name in flat:
        raise RuntimeError('two folder paths flatten to the same file name: '
                           + flat[local_name] + ' and ' + p + ' -> ' + local_name)
    flat[local_name] = p

PDFS = []
for local_name, p in sorted(flat.items()):
    dest = os.path.join(CRF_DIR, local_name)
    with folder_in.get_download_stream(p) as s, open(dest, 'wb') as f:
        shutil.copyfileobj(s, f)  # always overwrite - folder content may have changed
    PDFS.append(dest)

_keys = [doc_key(p) for p in PDFS]
assert len(set(_keys)) == len(_keys),     'doc_key collision among input PDFs (near-identical names) - rename the colliding files'
assert CODE_SUBPATH.strip('/') not in _keys,     'a PDF resolves to doc_key "' + CODE_SUBPATH + '" - its artifacts would mingle '     'with the module subpath; rename the file'
print(len(PDFS), 'pdf(s) staged')
for p in PDFS:
    print('  ', os.path.basename(p))


In [ ]:
# Stage 0 (pure Python, no LLM): cluster every page by structural layout and
# pick representative pages. The clusterer (generic_profile) is word-blind:
# each line becomes a typography/geometry token, tokens the document repeats
# on nearly every page (its own header/footer chrome) are discovered and
# down-weighted, pages are grouped by weighted-Jaccard similarity, and the
# similarity threshold theta is selected PER DOCUMENT by stability - no
# corpus-tuned constants. Encrypted or scanned (no text layer) PDFs are
# flagged here and never spend LLM budget. One corrupt PDF must not sink the
# batch: it is reported and skipped (no clusters.json -> the induction cell
# records it as skipped).
for _pdf in PDFS:
    try:
        _m = stage0_cluster.run(_pdf)
    except Exception as _e:
        print(f"{os.path.basename(_pdf)[:58]:58s} stage0 FAILED: {_e!r}")
        continue
    _flag = '' if _m.get('status') == 'ok' else '  [' + _m['status'] + ' - will skip induction]'
    _theta = ' theta*=%.2f' % _m['theta'] if _m.get('theta') is not None else ''
    print(f"{_m['file'][:58]:58s} pages={_m['pages']:5d} clusters={_m['n_clusters']:3d} "
          f"reps={len(_m['representative_pages_1based']):3d}{_theta}{_flag}")


In [ ]:
# LLM transport: Dataiku LLM Mesh (same get_llm/new_completion conventions as
# the ECS generation recipes). Plain-text completion per call, bounded retries.
import time

client = dataiku.api_client()
project = client.get_default_project()
llm_id = LLM_ID or project.get_variables()['local'].get('default_llm_model')
assert llm_id, 'set LLM_ID or the project variable default_llm_model'
llm = project.get_llm(llm_id)

def call_mesh(prompt, retries=2, backoff_s=15):
    last = None
    for attempt in range(retries + 1):
        try:
            comp = llm.new_completion()
            try:
                comp.settings.update(COMPLETION_SETTINGS)
            except Exception as e:
                # settings shape varies across mesh/provider versions; warn ONCE -
                # without maxOutputTokens the provider default cap may truncate
                # long generated programs (which then fail gates and burn budget)
                if not getattr(call_mesh, '_settings_warned', False):
                    call_mesh._settings_warned = True
                    print('WARNING: completion settings not applied (' + repr(e)
                          + '); mesh defaults in effect')
            comp.with_message(prompt)
            resp = comp.execute()
            if getattr(resp, 'success', False) and (resp.text or '').strip():
                return resp.text
            last = RuntimeError('unsuccessful or empty completion')
        except Exception as e:
            last = e
        if attempt < retries:
            time.sleep(backoff_s * (attempt + 1))
    raise RuntimeError('LLM Mesh call failed after ' + str(retries + 1) + ' attempts: ' + repr(last))

print('LLM Mesh id:', llm_id)


In [ ]:
# The induction loop. rci.induce_document is the SAME controller validated by
# stop_policy_test.py locally; only the transport (call_mesh) is Dataiku-specific.
# Stop rules: converged (clean audit) / plateau (no improvement between two
# consecutive versions) / budget (MAX_VERSIONS) - best version wins.
# A per-document failure becomes a summary row, never a lost batch (this can be
# a multi-hour paid run). Summary file is induction_summary_<tag>.json - the
# 'cli_' prefix is reserved for the local CLI driver so runs cannot be confused.
import json

tag = rci.slug(llm_id)
summary = []
for _pdf in PDFS:
    _key = doc_key(_pdf)
    _outdir = os.path.join(OUT_DIR, _key)
    print('===', _key)
    try:
        _meta = rci.doc_meta(_outdir)
        _status = _meta.get('status', 'ok')
        if _status != 'ok':
            summary.append({'doc': _key, 'status': 'skipped_' + _status})
            print('    skipped:', _status)
            continue
        _prompt = codegen.build_codegen_prompt(_pdf, _outdir)
        with open(os.path.join(_outdir, 'codegen_prompt.txt'), 'w', encoding='utf-8') as f:
            f.write(_prompt)
        _best, _trail, _stop, _versions = rci.induce_document(
            call_mesh, llm_id, tag, _pdf, _outdir, _prompt, MAX_VERSIONS)
        _row = rci.finalize_document(_key, _pdf, _outdir, tag, _best, _trail, _stop, _versions)
        if _meta.get('text_layer_pct', 100) < 100:
            # partially scanned book: its no-text pages are unreachable (OCR out
            # of scope) - surface that in the summary instead of hiding it
            _row['text_layer_pct'] = _meta['text_layer_pct']
    except Exception as _e:
        _row = {'doc': _key, 'status': 'error', 'error': repr(_e)}
    summary.append(_row)
    print('    ->', _row)

with open(os.path.join(OUT_DIR, 'induction_summary_' + tag + '.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=1)

import pandas as pd
pd.DataFrame(summary)


In [ ]:
# Persist all artifacts to the output managed folder, mirroring the local
# out/ tree: <doc_key>/clusters.json, rep_p*.txt, codegen_prompt.txt,
# codegen_reply_*<n>.py, codegen_trail_*.json, generated_extractor_*.py,
# fields_codegen_*.csv, plus induction_summary_*.json at the root.
# Scope: THIS run's documents only. A warm work dir may hold artifacts of
# documents an earlier session processed but the current DOC_FILTER excludes -
# re-uploading those would contradict the run scoping FETCH_CELL guarantees.
folder_out = dataiku.Folder(OUTPUT_FOLDER)
_run_keys = {doc_key(_p) for _p in PDFS}
uploaded = 0
for _root, _dirs, _files in os.walk(OUT_DIR):
    for _fn in _files:
        if _fn.endswith('.png') and not UPLOAD_PAGE_PNGS:
            continue
        _full = os.path.join(_root, _fn)
        _rel = os.path.relpath(_full, OUT_DIR).replace(os.sep, '/')
        _top = _rel.split('/')[0]
        if '/' in _rel and _top not in _run_keys:
            continue  # another session's document dir
        with open(_full, 'rb') as f:
            folder_out.upload_stream(_rel, f)
        uploaded += 1
print('uploaded', uploaded, 'file(s) to folder', OUTPUT_FOLDER,
      'for', len(_run_keys), 'document(s)')


In [ ]:
# OPTIONAL - name->OID mapping via the form-first funnel (oid_mapping.py):
#   1. form scoping   partial_ratio >= 70 (production's own convention,
#                     review_table.get_standard_crf)
#   2. field in form  token_sort_ratio >= 85 against the scoped rows only
#   3. LLM ranker     EVERY candidate-bearing pair is judged via LLM Mesh -
#                     pick one of the listed OIDs or refuse. String scores
#                     generate candidates; they never certify them.
# Unmapped pairs are safe by design: production writes LLM-generated rules
# from the names alone. Measured on the ground-truth book: 94% of what it
# maps is correct; coverage is bounded by library breadth, not this logic.
# Needs rapidfuzz. CAVEAT: _norm strips every non-ASCII character, so on
# non-Latin documents (or a non-Latin library) mapping coverage will be ~0%.
if RUN_OID_MAPPING:
    import csv
    import re

    import pandas as pd

    sys.modules.pop('oid_mapping', None)  # same rerun hygiene as the import cell
    import oid_mapping
    assert os.path.abspath(oid_mapping.__file__).startswith(
        os.path.abspath(MODULES_DIR) + os.sep), 'oid_mapping imported from outside the bundle'

    def _norm(s):
        s = re.sub(r'\(.*?\)', ' ', str(s or '').lower())
        s = re.sub(r'[^a-z0-9 ]+', ' ', s)
        return re.sub(r'\s+', ' ', s).strip()

    _lib_df = dataiku.Dataset(ECS_INDEX_DATASET).get_dataframe()
    _need = {'form_field_value', 'variable_name'}
    assert _need <= set(_lib_df.columns), ECS_INDEX_DATASET + ' must have columns ' + str(_need)
    _lib_df = _lib_df.fillna('')  # NaN is truthy - without this it becomes the string 'nan'
    _lib = []
    for _, _r in _lib_df.iterrows():
        _fv = str(_r.get('form_field_value') or '').strip()
        _vn = str(_r.get('variable_name') or '').strip()
        _fnorm = _norm(_fv)
        if not (_fv and _vn and _fnorm):  # drop rows whose label normalizes away
            continue
        _lib.append({'field': _fnorm, 'form': _norm(_r.get('form_name', '')),
                     'oid': _vn, 'field_raw': _fv,
                     'form_raw': str(_r.get('form_name', '')).strip()})
    print('library entries:', len(_lib))
    _formless = sum(1 for _e in _lib if not _e['form'])
    if _formless > len(_lib) // 2:
        # without form names layer-1 scoping matches nothing -> universal
        # unmapped that LOOKS like safe abstention but is a dataset problem
        print('WARNING:', _formless, 'of', len(_lib), 'library rows have no '
              'form_name - form scoping will unmap nearly everything')

    _done = _skipped = 0
    for _key in sorted(os.listdir(OUT_DIR)):
        _src = os.path.join(OUT_DIR, _key, 'fields_codegen_' + tag + '.csv')
        if not os.path.isdir(os.path.join(OUT_DIR, _key)):
            continue
        if not os.path.isfile(_src):
            _skipped += 1
            continue
        with open(_src, encoding='utf-8') as f:
            _pairs = sorted({(_norm(r['form_name']), _norm(r['field_name']))
                             for r in csv.DictReader(f)})
        try:
            # one Mesh failure must not sink the batch (rank_cases commits
            # nothing on failure, so there is no half-ranked state to persist)
            _results = oid_mapping.map_pairs(_pairs, _lib, llm=call_mesh)
        except Exception as _e:
            print(f'{_key[:52]:52s} mapping FAILED: {_e!r}')
            continue
        _rows = [{'form_name': r.form, 'field_name': r.field, 'status': r.status,
                  'oid': r.oid or '', 'via': r.via if r.status == 'mapped' else '',
                  'n_candidates': len(r.candidates)}
                 for r in _results]
        pd.DataFrame(_rows).to_csv(os.path.join(OUT_DIR, _key, 'oid_mapping_' + tag + '.csv'),
                                   index=False)
        _done += 1
        _mapped = [r for r in _results if r.status == 'mapped']
        _by = {v: sum(1 for r in _mapped if r.via == v)
               for v in sorted({r.via for r in _mapped})}
        print(f'{_key[:52]:52s} pairs={len(_pairs):5d} mapped={len(_mapped):4d} '
              f'({100 * len(_mapped) // max(1, len(_pairs))}%) by={_by}')
    print('mapped', _done, 'document(s);', _skipped,
          'dir(s) had no fields_codegen_' + tag + '.csv (check tag/run)')
    print('re-run the upload cell to persist the mapping CSVs')
else:
    print('RUN_OID_MAPPING is False - skipped')


## Reading the outputs

Per document (in the output folder, under `<doc_key>/`):

| artifact | meaning |
|---|---|
| `clusters.json` | stage-0 layout clusters, representative pages, selected `theta`, `status` |
| `rep_p<N>.txt` | the representative page dumps the LLM saw (text + geometry) |
| `codegen_prompt.txt` | the exact induction prompt |
| `codegen_reply_<tag>_<n>.py` | every parser version the LLM wrote |
| `codegen_reply_<tag>_confirm.txt` | the coverage-confirmation reply, if that round ran |
| `codegen_trail_<tag>.json` | per-version metrics/problems/audit + `stop_reason` + `best_version` |
| `generated_extractor_<tag>.py` | the accepted (best) parser |
| `fields_codegen_<tag>.csv` | final `form_name, field_name, page` extraction |
| `oid_mapping_<tag>.csv` | optional name->OID funnel result: status / oid / via (mapping cell) |

Summary `status` values: `ok` (clean audit, no warnings) / `ok_with_warnings`
(soft quality signals; document may legitimately violate them) /
`ok_audit_issues` (best version still has known page-level issues - review) /
`ok_unaudited` (audit errored; parser passed gates but was never page-verified) /
`needs_manual_template` (every version hard-failed - human review) /
`export_failed` (best parser accepted but the final replay crashed - re-run) /
`error` (unexpected per-document failure; see the cell output for the traceback) /
`skipped_encrypted`, `skipped_no_text_layer`, `skipped_no_pages`,
`skipped_missing_stage0` (stage 0 refused or failed on the PDF; OCR is out of
scope by design - fail loudly, never guess). A row may additionally carry
`text_layer_pct` when the book is PARTIALLY scanned (>=20% text pages proceed,
but the scanned pages are unreachable and their fields cannot appear in the
output - review such documents).

**Not in this notebook**: ground-truth evaluation (no annotated truth exists on
the Dataiku side; the local repo has `eval_form_field.py` for the one document
with printed OIDs) and the production OID-assignment step (form-scoped
candidates + LLM ranking - designed separately; the mapping cell here is the
lexical baseline only).
